In [11]:
#importation de base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns",None)

In [ ]:
#Chargement de la dataset CSV et lecture du header
DATA_PATH = "../data/data.csv"
df = pd.read_csv(DATA_PATH,encoding = "latin1") #convertir en encodage utf8
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/01/2010 08:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/01/2010 08:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/01/2010 08:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/01/2010 08:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/01/2010 08:26,3.39,17850.0,United Kingdom


In [ ]:
#info sur la donnée (premeire analyse de qualité)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [ ]:

df.describe(include="all")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
count,541909,541909,540455,541909.000000,541909,541909.000000,406829.000000,541909
unique,25900,4070,4223,NaN,23260,NaN,NaN,38
top,573585,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,10/31/2011 14:41,NaN,NaN,United Kingdom
freq,1114,2313,2369,NaN,1114,NaN,NaN,495478
mean,NaN,NaN,NaN,9.552250,NaN,4.611114,15287.690570,NaN
std,NaN,NaN,NaN,218.081158,NaN,96.759853,1713.600303,NaN
min,NaN,NaN,NaN,-80995.000000,NaN,-11062.060000,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000,NaN,1.250000,13953.000000,NaN
50%,NaN,NaN,NaN,3.000000,NaN,2.080000,15152.000000,NaN
75%,NaN,NaN,NaN,10.000000,NaN,4.130000,16791.000000,NaN


In [23]:
#premier état sur la qualité des données
missing = (
    df.isna()
      .mean()
      .mul(100)
      .round(2)
      .reset_index()
      .rename(columns={"index":"column",0:"missing_pct(in %)"})
)
missing

,column,missing_pct(in %)
0,InvoiceNo,0.00
1,StockCode,0.00
2,Description,0.27
3,Quantity,0.00
4,InvoiceDate,0.00
5,UnitPrice,0.00
6,CustomerID,24.93
7,Country,0.00


In [24]:
df.duplicated().sum()

np.int64(5268)

In [26]:
numeric_cols = ["Quantity","UnitPrice"]
df[numeric_cols].describe(percentiles=[0.01,0.05,0.95,0.99])

,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
1%,-2.000000,0.190000
5%,1.000000,0.420000
50%,3.000000,2.080000
95%,29.000000,9.950000
99%,100.000000,18.000000
max,80995.000000,38970.000000


In [27]:
data_quality = pd.DataFrame({
    "missing_pct": df.isna().mean() *100,
    "n_unique" : df.nunique(),
    "dtype": df.dtypes
})
data_quality

,missing_pct,n_unique,dtype
InvoiceNo,0.000000,25900,object
StockCode,0.000000,4070,object
Description,0.268311,4223,object
Quantity,0.000000,722,int64
InvoiceDate,0.000000,23260,object
UnitPrice,0.000000,1630,float64
CustomerID,24.926694,4372,float64
Country,0.000000,38,object


# EDA Level 1 - Decouverte clé
-le dataset contient plus de 540k transactions, avec une structure typique d'e-commerce
-La colonne CustumerID presente pres de 25% de valeur manquantes, indiquant les achats anononymes.
-Des quantités et prix negatifs ou extremes suggèrent la présence de retours et d'erreurs de saisie.
-Le Royaume uni concentre la majorité des transactions.
-Des Doublons exactes sont présent et devront ètre supprimés

ces observations guideront les choix de nettoyage effectués  à l'étape suivante 